rumz
# Notebook for unsupervised ML in HSI Ice Cores
### Feb 25
### V 0.2

This notebook processes hyperspectral images of ice and runs unsupervised ML for clustering.
To run:
- ensure datasetPath points to the correct directory file
- Insert DataID for whichever core to be run

In [1]:
import warnings
import sys
import os

# Add the processing folder to the Python path
processing_path = os.path.abspath(os.path.join('..', 'processing'))
if processing_path not in sys.path:
    sys.path.append(processing_path)

#Importing Python Libraries and Packages

import pandas as pd
import numpy as np
from spectral import *
import matplotlib.pyplot as plt

import set_icespec_path as tool
icespec_code_path=tool.add_icespec_code_dir()
tool.add_icespec_code_dir()
from icespec import icespec_process1_module as processice
from icespec import savePNG

In [ ]:
datasetPath = '/data/icecores/HSI_2024.csv' #path to csv file on computer or server


savePath = '/data/icecores/Output/HSIUM-101/HSIUM-101_Prelim.png' #png save path

DataID = "HSIUM-104" #insert DataID from excel file inside quotation marks

Debug = True

dataPath, top_depth, bottom_depth, core_length = processice.getDataPath(datasetPath, DataID)

%matplotlib ipympl
data_nparr, w_ref, d_ref, fwhm = processice.getEnviData(dataPath)

In [3]:
dataSize = [1500, 6600, 200, 900] #fill in based on interactive plot
stdSize = [530, 820, 200, 900] #standard sizing

In [ ]:
%matplotlib inline
trimmedData = processice.trim(data_nparr, w_ref, d_ref, dataSize, stdSize, Debug=True) #return arr of 3 trimmed
correctedData = processice.reflectance(trimmedData, Debug)
filtered_img = processice.remove_bands(correctedData, trimmedData[1])
previewed_img = processice.previewImg(filtered_img, "Filtered Image")
inpainted_img = processice.bubbles(filtered_img,  fwhm)
flattened_img = processice.flatten_data(inpainted_img)
rematrix_img = processice.resize_matrix(flattened_img, trimmedData, pc_num = 5) #pc_num can be changed
eq_img = processice.equalize(rematrix_img)

(m, c) = kmeans(rematrix_img, 5, 50)
(x, y) = kmeans(inpainted_img, 5, 50)

In [ ]:
fig = plt.figure(figsize=(3, 5))
plt.imshow(m)
plt.colorbar(label='Cluster Index')
plt.title('Clustered Image')
plt.show()

In [ ]:
fig = plt.figure(figsize=(3, 5))
plt.imshow(x)
plt.colorbar(label='Cluster Index')
plt.title('Clustered Image')
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Define number of components (clusters)
num_clusters = 4  

hierarchical = AgglomerativeClustering(n_clusters=num_clusters, linkage='ward')
labels_hierarchical = hierarchical.fit_predict(flattened_img)

class_map_hierarchical = labels_hierarchical.reshape(5100, 700)

# Plot results
plt.imshow(class_map_hierarchical, cmap='magma')
plt.colorbar(label='Cluster Index')
plt.title('Hierarchical Clustering on Hyperspectral Data')
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch


linkage_matrix = sch.linkage(flattened_img[:1000], method='ward')

plt.figure(figsize=(10, 5))
sch.dendrogram(linkage_matrix, truncate_mode='level', p=10)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.show()

# TODO:
- Retry hierarchal cluster w/o full image load
- Comments
- Check for hardcoding
- PyQtGraph
